# Setup

In [1]:
# !pip install selenium webdriver-manager pandas beautifulsoup4 lxml

In [7]:
import os
import time
import pandas as pd
import re
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from webdriver_manager.chrome import ChromeDriverManager

# Config

In [3]:
BASE_URL = "https://haims2.doh.go.th/"
USERNAME = "com_cu"
PASSWORD = "abab1212!"

TARGET_INCIDENT_ID = "1158950"

WAIT_TIME = 10

# Start Browser

In [6]:
options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

wait = WebDriverWait(driver, WAIT_TIME)

driver.get(BASE_URL)

# Login

In [ ]:
wait.until(lambda d: d.execute_script("return document.readyState") == "complete")

username_input = wait.until(
    EC.element_to_be_clickable((By.ID, "edit-name"))
)

password_input = wait.until(
    EC.element_to_be_clickable((By.ID, "edit-pass"))
)

login_button = wait.until(
    EC.element_to_be_clickable((By.ID, "edit-submit"))
)

username_input.click()
username_input.clear()
username_input.send_keys(USERNAME)

password_input.click()
password_input.clear()
password_input.send_keys(PASSWORD)

# time.sleep(3)

login_button.click()

# time.sleep(3)

wait.until(
    EC.presence_of_element_located((By.ID, "summary_main_table"))
)

print("Login Success")

# Navigate to Accident Page

In [ ]:
accident_button = wait.until(
    EC.element_to_be_clickable(
        (By.XPATH, '//*[@id="summary_main_table"]/tbody/tr[2]/td[1]/a')
    )
)

accident_button.click()

print("Opened Accident Page")

# Find Incident ID

## Get Total Page

In [ ]:
page_info = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, '//*[@id="caselist"]/caption/div/div[3]/span[5]')
    )
).text

print("Page text:", page_info)

# extract number
total_pages = int(re.search(r'\d+$', page_info).group())

print("Total pages:", total_pages)

## Loop Find Incident ID

In [ ]:
found = False

for page in range(total_pages):

    print(f"Checking page {page+1}/{total_pages}")

    rows = driver.find_elements(By.CSS_SELECTOR,"tr[validrow='true']")

    for row in rows:

        incident_id = row.find_element(By.CSS_SELECTOR,"td.id").text.strip()

        print("checking:", incident_id)

        if incident_id == str(TARGET_INCIDENT_ID):
            target_row = row
            found = True
            break

    if found:
        break

    # === แก้ไข 1: ดักจับหน้าสุดท้าย ถ้าเป็นหน้าสุดท้ายแล้วไม่ต้องกด Next ===
    if page == total_pages - 1:
        print("Reached the last page, but ID not found.")
        break

    # remember first row id
    old_first_id = rows[0].find_element(By.CSS_SELECTOR,"td.id").text

    print("CLICK NEXT PAGE")

    next_button = wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, "input.nextPage")
        )
    )

    # === แก้ไข 2: ใช้ JavaScript Click ทะลวงกดปุ่มให้ชัวร์ว่าติด 100% ===
    driver.execute_script("arguments[0].click();", next_button)

    # === แก้ไข 3: เพิ่ม Try-Except ดักรอหน้าเว็บโหลด ป้องกันการพังถ้าระบบโหลดช้า ===
    try:
        # เช็คให้ชัวร์ว่าตารางโหลดมาแล้ว (len > 0) และ ID แถวแรกไม่เหมือนหน้าเดิม
        wait.until(
            lambda d: len(d.find_elements(By.CSS_SELECTOR,"tr[validrow='true']")) > 0 and 
                      d.find_elements(By.CSS_SELECTOR,"tr[validrow='true']")[0].find_element(By.CSS_SELECTOR,"td.id").text != old_first_id
        )
        print("PAGE CHANGED")
    except Exception as e:
        print("Warning: รอเปลี่ยนหน้านานเกินไป เว็บอาจจะโหลดช้า หรือมีปัญหาขัดข้อง")

if not found:
    raise Exception("Incident ID not found")

print("Incident Found")

# Reaching Data

## Check Edit Button

In [ ]:
try:
    
    edit_button = target_row.find_element(By.XPATH, './/td[15]//a')
    
    print("Edit Button Found")

except:
    
    print("No edit button → program stop")
    driver.quit()

## Open Edit Page

In [ ]:
edit_button.click()

time.sleep(3)

# switch to new tab
driver.switch_to.window(driver.window_handles[-1])

# wait page load
wait.until(
    EC.presence_of_element_located((By.TAG_NAME, "form"))
)

print(driver.current_url)

time.sleep(3)

print("Incident form opened")

# Extract Sections

In [ ]:
html = driver.page_source
soup = BeautifulSoup(html,"html.parser")

project_id = TARGET_INCIDENT_ID

## Helper Function

### Universal Element Finder

In [ ]:
def find_element(key, tag=None):

    # search by name
    el = soup.find(tag or True, {"name": key})
    if el:
        return el

    # search by id
    el = soup.find(tag or True, {"id": key})
    if el:
        return el

    # search by class
    el = soup.find(tag or True, class_=key)
    if el:
        return el

    # search label tag
    label = soup.find("label", string=lambda x: x and key in x)
    if label:
        return label.find_next()

    # search text label (span/div)
    text_label = soup.find(string=lambda x: x and key in x)
    if text_label:
        return text_label.parent

    return None

### Universal Value Extractor

In [ ]:
def get_value(el):

    if not el:
        return None

    # ถ้ามี input อยู่ข้างใน
    inputs = el.find_elements(By.TAG_NAME, "input")
    if inputs:
        return inputs[0].get_attribute("value")

    # textarea
    textareas = el.find_elements(By.TAG_NAME, "textarea")
    if textareas:
        return textareas[0].get_attribute("value")

    # select
    selects = el.find_elements(By.TAG_NAME, "select")
    if selects:
        for opt in selects[0].find_elements(By.TAG_NAME, "option"):
            if opt.is_selected():
                return opt.text.strip()

    return el.text.strip()

In [ ]:
def get_attr_by_id(element_id, attr):
    try:
        el = driver.find_element(By.ID, element_id)
        return el.get_attribute(attr)
    except:
        return None

### Universal Driver Element Finder

In [ ]:
def get_by_key(key):
    
    try:
        el = driver.find_element(By.NAME, key)
        return el.get_attribute("value")
    except:
        pass

    try:
        el = driver.find_element(By.NAME, key)
        return get_value(el)
    except:
        pass

    try:
        el = driver.find_element(By.ID, key)
        return get_value(el)
    except:
        pass

    try:
        el = driver.find_element(By.CLASS_NAME, key)
        return get_value(el)
    except:
        pass

    return None

In [ ]:
def get_by_css(css):
    el = driver.find_element(By.CSS_SELECTOR, css)
    return get_value(el)

### Checkbox & Label Finder

In [ ]:
def get_checkbox_labels(name):

    results = []

    checkboxes = driver.find_elements(By.NAME, name)

    for cb in checkboxes:
        if cb.is_selected():

            cb_id = cb.get_attribute("id")

            label = driver.find_element(By.CSS_SELECTOR, f"label[for='{cb_id}']")
            results.append(label.text.strip())

    return results

## SECTION 0: Header

In [ ]:
def split_user_datetime(text):

    if not text:
        return None, None

    parts = text.rsplit(" ", 4)

    if len(parts) >= 5:
        user = parts[0]
        dt = " ".join(parts[1:])
        return user.strip(), dt.strip()

    return None, None

In [ ]:
userdata = driver.find_element(By.ID, "userdata").text
lines = userdata.split("\n")

create_text = lines[0].replace("สร้าง:", "").strip()
update_text = lines[1].replace("แก้ไข:", "").strip()

create_user, create_datetime = split_user_datetime(create_text)
update_user, update_datetime = split_user_datetime(update_text)

In [ ]:
metadata = driver.find_elements(By.CSS_SELECTOR, "#metadata .d-inline-block")

highway_reporter = get_value(metadata[0])
receiver = get_value(metadata[1])
sender = get_value(metadata[2])

In [ ]:
header = {
    "project_id": project_id,
    "create_datetime": create_datetime,
    "create_user": create_user,
    "update_datetime": update_datetime,
    "update_user": update_user,
    "highway_reporter": highway_reporter,
    "receiver": receiver,
    "sender": sender,
}

header_df = pd.DataFrame([header])

In [ ]:
header_df

## SECTION 1-2

In [ ]:
date = driver.find_element(By.ID,"date-mask").get_attribute("value")

hour = driver.find_element(By.ID,"hour") \
    .find_element(By.CSS_SELECTOR,"option:checked").text

minute = driver.find_element(By.ID,"minute") \
    .find_element(By.CSS_SELECTOR,"option:checked").text

incident_datetime = f"{date} {hour}:{minute}"

is_major_accident = 1 if driver.find_element(By.ID,"is_big_accident").is_selected() else 0

is_public_interest = 1 if driver.find_element(By.ID,"has_public_figure").is_selected() else 0

In [ ]:
section_1_2 = {
    "project_id": project_id,
    "incident_datetime": incident_datetime,
    "is_major_accident": is_major_accident,
    "is_public_interest": is_public_interest
}

section_1_2_df = pd.DataFrame([section_1_2])

In [ ]:
section_1_2_df

## SECTION 3

In [ ]:
is_new_construction_zone = 1 if driver.find_element(By.ID,"is_new_construct_road").is_selected() else 0

highway_district = driver.find_element(By.ID, "bureau") \
    .find_element(By.CSS_SELECTOR, "option[selected]").text
highway_subdistrict = driver.find_element(By.ID, "district").get_attribute("value")
highway_maintenance_unit = driver.find_element(By.ID, "depo").get_attribute("value")

highway_number = driver.find_element(By.ID, "route").get_attribute("value")
control_section = driver.find_element(By.ID, "controlsection").get_attribute("value")
kilopost = driver.find_element(By.ID, "kilometre").get_attribute("value").lstrip('0') + "+" + driver.find_element(By.ID, "metre").get_attribute("value")

roadinfo_text = driver.find_element(By.ID, "roadinfo").text
parts = [x.strip() for x in roadinfo_text.split(" / ")]
road_name = None
section_name = None
province = None
section_start_km = None
section_end_km = None
if len(parts) > 0:
    km_text = parts[-1] 
    
    km = re.findall(r'\d+\+\d+', km_text)
    if len(km) >= 2:
        section_start_km = km[0]
        section_end_km = km[1]
    elif len(km) == 1:
        section_start_km = km[0]
        
    if len(parts) >= 4:
        road_name = parts[0]
        section_name = parts[1]
        province = parts[-2] 
        
    elif len(parts) == 3:
        road_name = parts[0]
        province = parts[1]
        
    elif len(parts) == 2:
        road_name = parts[0]

road_feature = driver.find_element(By.ID, "road_char").text
road_status = driver.find_element(By.ID, "road_condition").text
lanes_count = driver.find_element(By.ID, "road_lane").get_attribute("value")

road_direction = driver.find_element(By.ID, "road_direction").text
median_type = driver.find_element(By.ID, "road_isle").text
traffic_type = driver.find_element(By.ID, "road_traffic").text
surface_type = driver.find_element(By.ID, "road_surface").text


In [ ]:
section_3 = {
    "project_id": project_id,

    "is_new_construction_zone": is_new_construction_zone,

    "highway_district": highway_district,
    "highway_subdistrict": highway_subdistrict,
    "highway_maintenance_unit": highway_maintenance_unit,

    "highway_number": highway_number,
    "control_section": control_section,
    "kilopost": kilopost,

    "road_name": road_name,
    "section_name": section_name,
    "province": province,
    "section_start_km": section_start_km,
    "section_end_km": section_end_km,

    "road_feature": road_feature,
    "road_status": road_status,
    "lanes_count": lanes_count,

    "direction": road_direction,
    "median_type": median_type,
    "traffic_type": traffic_type,
    "surface_type": surface_type,
}

section_3_df = pd.DataFrame([section_3])

In [ ]:
section_3_df

## SECTION 4

In [ ]:
road_horizontal = driver.find_element(By.ID, "char_horizontal").text
road_vertical = driver.find_element(By.ID, "char_vertical").text
intersection = driver.find_element(By.ID, "char_intersection").text
median_opening = driver.find_element(By.ID, "char_openacess").text
connector = driver.find_element(By.ID, "char_connect").text
special_area = driver.find_element(By.ID, "char_other").text

In [ ]:
section_4 = {
    "project_id": project_id,
    "road_horizontal": road_horizontal,
    "road_vertical": road_vertical,
    "intersection": intersection,
    "median_opening": median_opening,
    "connector": connector,
    "special_area": special_area,
}

section_4_df = pd.DataFrame([section_4])

In [ ]:
section_4_df

## SECTION 5

In [ ]:
try:
    control_span = wait.until(EC.presence_of_element_located((By.ID, "control")))
    
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", control_span)
    time.sleep(1) 
    
    driver.execute_script("arguments[0].click();", control_span)
    print("Clicked on 'control' span successfully.")
    
    time.sleep(2) 
    
except Exception as e:
    print(f"Cannot click the control span: {e}")

In [ ]:
selected_controls = get_checkbox_labels("control[]")

In [ ]:
section_5 = {
    "project_id": project_id,
    "speed_sign": 1 if "ป้ายจำกัดความเร็ว" in selected_controls else 0,
    "stop_sign": 1 if "ป้ายบังคับหยุด" in selected_controls else 0,
    "warning_sign": 1 if "ป้ายจราจรประเภทเตือนอื่นๆ" in selected_controls else 0,
    "traffic_light": 1 if "สัญญาณไฟจราจร" in selected_controls else 0,
    "flashing_light": 1 if "สัญญาณไฟกระพริบ" in selected_controls else 0,
    "road_marking": 1 if "เส้นเครื่องหมายจราจรบนผิวทาง" in selected_controls else 0,
    "no_overtake_zone": 1 if "เขตห้ามแซง" in selected_controls else 0,
    "no_parking_zone": 1 if "เขตห้ามจอด" in selected_controls else 0,
    "uncontrolled_crosswalk": 1 if "เป็นทางคนเดินข้ามถนนที่ไม่มีการควบคุม" in selected_controls else 0,
    "controlled_crosswalk": 1 if "เป็นทางคนเดินข้ามถนนที่มีการควบคุม" in selected_controls else 0,
    "pedestrian_bridge": 1 if "สะพานลอยคนเดินข้าม" in selected_controls else 0,
    "optical_speed_bar": 1 if "Optical Speed Bar" in selected_controls else 0,
    "rumble_strip": 1 if "Rumble Strip" in selected_controls else 0,
    "speed_camera": 1 if "กล้องตรวจจับความเร็ว" in selected_controls else 0,
    "guide_sign": 1 if "เครื่องหมายนำทาง" in selected_controls else 0,
    "no_control": 1 if "ไม่มีการควบคุมอย่างใดอย่างหนึ่ง" in selected_controls else 0,
}

section_5_df = pd.DataFrame([section_5])

In [ ]:
section_5_df

## SECTION 6-7

In [ ]:
def get_time_dropdown(key):
    """ฟังก์ชันเฉพาะสำหรับดึง Text จาก Dropdown (Select)"""
    try:
        try:
            el = driver.find_element(By.ID, key)
        except:
            el = driver.find_element(By.NAME, key)
            
        select = Select(el)
        return select.first_selected_option.text.strip()
    except:
        return ""

In [ ]:
traffic_map = {
    "1": "น้อยกว่า 500 เมตร",
    "2": "500 เมตร ถึง 1 กิโลเมตร",
    "3": "1 กิโลเมตร ถึง 2 กิโลเมตร",
    "4": "มากกว่า 2 กิโลเมตร",
    "5": "ไม่ติดสะสม"
}

traffic_congestion_no = get_attr_by_id("traffic_condition_parent", "data-value")
traffic_congestion = traffic_map.get(traffic_congestion_no)

start_hour = get_time_dropdown("traffic_condition_begin-hour")
start_min  = get_time_dropdown("traffic_condition_begin-minute")
end_hour   = get_time_dropdown("traffic_condition_end-hour")
end_min    = get_time_dropdown("traffic_condition_end-minute")
start_time = f"{start_hour}:{start_min}"
end_time = f"{end_hour}:{end_min}"

road_surface = driver.find_element(By.ID, "env_surface").text
road_condition = driver.find_element(By.ID, "env_status").text
weather = driver.find_element(By.ID, "env_weather").text
lighting = driver.find_element(By.ID, "env_light").text

In [ ]:
section_6_7 = {
    "project_id": project_id,
    "traffic_congestion": traffic_congestion,
    "start_time": start_time,
    "end_time": end_time,
    
    "road_surface": road_surface,
    "road_condition": road_condition,
    "weather": weather,
    "lighting": lighting,
}

section_6_7_df = pd.DataFrame([section_6_7])

In [ ]:
section_6_7_df

## SECTION 8: Vehicle

In [ ]:
gender_map = {
    "1": "ชาย",
    "2": "หญิง"
}

safety_map = {
    "1": "หมวกนิรภัย",
    "2": "เข็มขัดนิรภัย",
    "3": "ไม่ใช้",
    "4": "ไม่ระบุ"
}

injury_map = {
    "1": "ตาย ณ จุดเกิดเหตุ",
    "2": "ตาย ณ โรงพยาบาล",
    "3": "บาดเจ็บสาหัส",
    "4": "บาดเจ็บเล็กน้อย",
    "5": "ไม่ได้รับบาดเจ็บ",
    "6": "ไม่ระบุ"
}

drug_map = {
    "1": "มี",
    "2": "ไม่มี",
    "3": "ไม่ระบุ"
}

In [ ]:
vehicles = []

vehicle_blocks = soup.select("#vehiclelist li")

for v in vehicle_blocks:

    def get_value(name):
        el = v.select_one(f'input[name="{name}"]')
        return el["value"].strip() if el else None

    item = {
        "project_id": project_id,
        "vehicle_brand": get_value("vehicle_brand[]"),
        "color": get_value("vehicle_color[]"),
        "driver_name": get_value("vehicle_name[]"),
        "driver_citizen_id": get_value("vehicle_id_no[]"),
        "driver_age": get_value("vehicle_age[]"),
        "driver_gender": gender_map.get(get_value("vehicle_sex[]")),
        "safety_equipment": safety_map.get(get_value("vehicle_safety[]")),
        "injury_status": injury_map.get(get_value("vehicle_damage[]")),
    }

    vehicles.append(item)

section_8_df = pd.DataFrame(vehicles)

In [ ]:
section_8_df

## SECTION 9: Cause

In [ ]:
categories = {
    "100": "ด้านสภาพแวดล้อม",
    "200": "ด้านยานพาหนะ",
    "300": "ด้านผู้ขับขี่",
    "400": "ด้านคนเดินเท้า"
}

In [ ]:
causes = []

for code, category_name in categories.items():

    items = soup.select(f"#causelist-{code}-items li")

    for i, item in enumerate(items, start=1):

        cause = item.text.split("\n ")[1].strip()

        data = {
            "project_id": project_id,
            "category": category_name,
            "rank": i,
            "cause": cause
        }

        causes.append(data)

section_9_df = pd.DataFrame(causes)

In [ ]:
section_9_df

## SECTION DIAGRAM

In [ ]:
crash_id = driver.find_element(By.ID, "crash").text
inci_address = soup.select_one("#address").get_text(strip=True).replace("สถานที่:", "")
gps_loc = soup.select_one("#position").get_text(strip=True).split(', ')
gps_lat = gps_loc[0]
gps_lon = gps_loc[1]
inci_desc = soup.select_one("#summary").get_text(strip=True)

In [ ]:
diagram = {
    "project_id": project_id,
    "crash_id": crash_id,
    "incident_address": inci_address,
    "gps_lat": gps_lat,
    "gps_lng": gps_lon,
    "incident_description": inci_desc,
}

diagram_df = pd.DataFrame([diagram])

In [ ]:
diagram_df

## Save .csv

In [ ]:
folder = f"{project_id}_dataset"
os.makedirs(folder, exist_ok=True)

header_df.to_csv(f"{folder}/header.csv", index=False)
section_1_2_df.to_csv(f"{folder}/section_1_2.csv", index=False)
section_3_df.to_csv(f"{folder}/section_3.csv", index=False)
section_4_df.to_csv(f"{folder}/section_4.csv", index=False)
section_5_df.to_csv(f"{folder}/section_5.csv", index=False)
section_6_7_df.to_csv(f"{folder}/section_6_7.csv", index=False)
section_8_df.to_csv(f"{folder}/section_8.csv", index=False)
section_9_df.to_csv(f"{folder}/section_9.csv", index=False)
diagram_df.to_csv(f"{folder}/section_diagram.csv", index=False)